## Phase 9

In [ ]:
# ============================================
# PHASE 9 — LOAD PHASE 7 RESULTS
# Change RUN to "B" later to repeat everything for Run B.
# ============================================
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.metrics import mean_squared_error, median_absolute_error, r2_score

RUN = "A"
d = joblib.load(f"results/phase7_{RUN}.joblib")
Xt_tr, Xt_va, Xt_te = d["Xt_tr"], d["Xt_va"], d["Xt_te"]
y_train, y_val, y_test = d["y_train"], d["y_val"], d["y_test"]
prep_all, best, ols = d["prep_all"], d["best"], d["ols"]
baseline_value, tune, outlier_tr = d["baseline_value"], d["tune"], d["outlier_tr"]
names = prep_all.get_feature_names_out()

def rmse(a, p):
    return float(np.sqrt(mean_squared_error(a, p)))

def predict(name, X):
    if name == "baseline":
        return np.full(X.shape[0], baseline_value)
    if name == "ols":
        return ols.predict(X)
    score, model, extra = best[name]
    if name == "enet_rescaled":
        coef, intercept = extra
        return np.asarray(X @ coef).ravel() + intercept
    return model.predict(X)

def coef_of(name):
    if name == "ols":
        return ols.coef_
    if name == "enet_rescaled":
        return best[name][2][0]
    return best[name][1].coef_

# ---- Score the TEST set once ----
rows = []
for name in ["baseline", "ols", "ridge", "lasso", "enet_naive", "enet_rescaled"]:
    p = predict(name, Xt_te)
    yt = np.asarray(y_test)
    rows.append({
        "Model": name,
        "Test RMSE (log)": rmse(yt, p),
        "Test R2": r2_score(yt, p),
        "Median $ error": float(np.median(np.abs(np.exp(yt) - np.exp(p)))),
        "Nonzero coefs": None if name == "baseline" else int((coef_of(name) != 0).sum()),
    })
test_results = pd.DataFrame(rows)
display(test_results.round(4))
test_results.to_csv(f"results/run_{RUN}.csv", index=False)